# Notebook version of `cnn_from_scratch.py`

This notebook mirrors the Python script and keeps output-interpretation notes in the script comments.
Run the code cell below to reproduce outputs.

In [ ]:
# Output Guide:
# - Prints input, feature-map, and pooled-map shapes, plus one final logit.
# - Correctness check: dimensions should reduce after convolution/pooling as expected.
# - The logit is an unbounded score (not a probability); its sign/magnitude indicates class tendency.
"""
Convolutional Neural Network (CNN) — NumPy From Scratch
========================================================
Demonstrates the core forward-pass primitives of a CNN:
  1. 2-D convolution  (single kernel, no padding)
  2. ReLU activation
  3. 2×2 max-pooling  (stride 2)
  4. Flatten + dense readout (logit)

No training / backprop — this file focuses on the forward mechanics.
"""

import numpy as np


# ── Reproducibility ───────────────────────────────────────────────────────────
rng = np.random.default_rng(42)


# ── Synthetic image ───────────────────────────────────────────────────────────
def make_image(size: int = 8) -> np.ndarray:
    """
    Create an 8×8 grayscale image with a bright vertical bar
    plus a small amount of Gaussian noise.
    """
    img = np.zeros((size, size))
    img[2:6, 3:5] = 1.0                               # vertical bar
    img += rng.normal(0, 0.05, img.shape)              # add noise
    return img


# ── CNN primitives ────────────────────────────────────────────────────────────
def conv2d(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Perform a valid 2-D cross-correlation (convolution without padding).

    Parameters
    ----------
    image  : 2-D input array of shape (H, W)
    kernel : 2-D filter of shape (kH, kW)

    Returns
    -------
    feature_map : 2-D array of shape (H - kH + 1, W - kW + 1)
    """
    kH, kW = kernel.shape
    out_H = image.shape[0] - kH + 1
    out_W = image.shape[1] - kW + 1

    feature_map = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            # Element-wise multiply the patch with the kernel and sum
            feature_map[i, j] = np.sum(image[i:i + kH, j:j + kW] * kernel)

    return feature_map


def relu(z: np.ndarray) -> np.ndarray:
    """Apply ReLU activation element-wise: max(0, z)."""
    return np.maximum(0, z)


def max_pool2d(feature_map: np.ndarray, pool_size: int = 2) -> np.ndarray:
    """
    Apply 2-D max-pooling with a square window and stride equal to pool_size.

    Parameters
    ----------
    feature_map : 2-D array of shape (H, W)
    pool_size   : integer size of the pooling window (default 2)

    Returns
    -------
    pooled : 2-D array of shape (H // pool_size, W // pool_size)
    """
    H, W = feature_map.shape
    out_H = H // pool_size
    out_W = W // pool_size

    pooled = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            row_start = i * pool_size
            col_start = j * pool_size
            patch = feature_map[row_start:row_start + pool_size,
                                 col_start:col_start + pool_size]
            pooled[i, j] = patch.max()

    return pooled


def dense_readout(flat_features: np.ndarray, out_dim: int = 1) -> np.ndarray:
    """
    A single fully-connected layer with random weights (no training).
    Used here just to show how the flattened features feed into a classifier.
    """
    W = rng.normal(size=(flat_features.shape[0], out_dim))
    return flat_features @ W


# ── Main forward pass ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Build a synthetic image
    image = make_image()
    print(f"Input image shape : {image.shape}")

    # 2. Define a vertical-edge-detection kernel (Prewitt-style)
    kernel = np.array([
        [-1,  0,  1],
        [-1,  0,  1],
        [-1,  0,  1],
    ], dtype=float)

    # 3. Convolution → detects vertical edges
    feature_map = conv2d(image, kernel)
    print(f"Feature map shape : {feature_map.shape}")

    # 4. ReLU → suppress negative activations
    activated = relu(feature_map)

    # 5. Max-pooling → spatial downsampling
    pooled = max_pool2d(activated, pool_size=2)
    print(f"Pooled map shape  : {pooled.shape}")

    # 6. Flatten → feed into dense readout
    flat = pooled.ravel()
    logit = dense_readout(flat)
    print(f"Logit (pre-sigmoid) : {logit[0]:.4f}")

